# Preprocessing and Feature Extraction
Notebook ini mengekstrak fitur warna HSI dari dataset gambar mentah dan menyimpannya ke dalam file CSV.

In [ ]:
import os
import glob
import pandas as pd
import sys
sys.path.append(os.path.join('..', 'src'))
from image_utils import load_and_preprocess_image
from feature_extraction import extract_features

def extract_features_from_directory(base_dir):
    data = []
    for class_name in os.listdir(base_dir):
        class_path = os.path.join(base_dir, class_name)
        if not os.path.isdir(class_path):
            continue
            
        label = class_name.lower()
        if label == 'unripe':
            label = 'mentah'
        elif label == 'ripe':
            label = 'matang'
        elif label in ['overripe', 'overipe']:
            label = 'terlalu matang'
            
        image_paths = glob.glob(os.path.join(class_path, '*.*'))
        print(f"Processing {len(image_paths)} images for class {label} in {base_dir}...")
        
        for img_path in image_paths:
            if not (img_path.lower().endswith('.jpg') or img_path.lower().endswith('.png') or img_path.lower().endswith('.jpeg')):
                continue
                
            img = load_and_preprocess_image(img_path)
            if img is None:
                continue
                
            features = extract_features(img)
            fruit_type = os.path.basename(img_path).split('_')[0]
            row = [fruit_type] + features + [label]
            data.append(row)
            
    columns = ['fruit', 'mean_h', 'std_h', 'mean_s', 'std_s', 'mean_i', 'std_i', 'label']
    df = pd.DataFrame(data, columns=columns)
    return df

train_dir = os.path.join('..', 'data', 'raw', 'Train')
test_dir = os.path.join('..', 'data', 'raw', 'Test')
out_dir = os.path.join('..', 'data', 'processed')
os.makedirs(out_dir, exist_ok=True)

print("Extracting training features...")
df_train = extract_features_from_directory(train_dir)
df_train.to_csv(os.path.join(out_dir, 'train_features.csv'), index=False)

print("Extracting testing features...")
df_test = extract_features_from_directory(test_dir)
df_test.to_csv(os.path.join(out_dir, 'test_features.csv'), index=False)

print("Extraction complete!")